# Bước 5: Huấn luyện Propositionizer (Replication Figure 5)
Notebook này tái hiện lại chuẩn xác quy trình huấn luyện mô hình **Propositionizer** từ bài báo *Dense X Retrieval*:
- **Mô hình gốc:** `google/flan-t5-base` (250M tham số, giữ nguyên 100% pretrained word embeddings).
- **Dữ liệu huấn luyện:** 4,000 cặp `(Passage -> Danh sách Propositions)` trích xuất trực tiếp từ FactoidWIKI bằng DuckDB.
- **Hyperparameters của tác giả:**
  - Optimizer: `AdamW`
  - Learning rate: `1e-4`, Weight decay: `1e-4`
  - Effective batch size: `64` (`batch_size=4` x `gradient_accumulation_steps=16`)
  - Epochs: `3`
- **Độ chính xác số học:** Huấn luyện bằng `fp32` (`fp16=False`) để loại bỏ 100% lỗi Underflow / Loss = 0.0 / NaN kinh điển của dòng họ T5.
- **Đánh giá (Metric):** Đo điểm **BERTScore F1** (sử dụng `roberta-large`) trên tập Dev để đối chiếu với ngưỡng 0.80 - 0.82 trong Figure 5.

In [1]:
# Ép sử dụng GPU 0 độc lập để tránh hoàn toàn lỗi DataParallel multi-GPU làm hỏng gradient
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install transformers datasets accelerate duckdb bert-score evaluate tabulate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


## 0. Nạp HuggingFace Token

In [ ]:
from huggingface_hub import login

# ĐIỀN TOKEN CỦA BẠN VÀO ĐÂY
TOKEN_CUA_BAN = 'hf_xxxx' 
login(token=TOKEN_CUA_BAN)
os.environ["HF_TOKEN"] = TOKEN_CUA_BAN
print("Đã nạp HF Token thành công!")

Đã nạp HF Token thành công!


## 1. Trích xuất Cặp Dữ liệu Huấn luyện từ FactoidWiki bằng DuckDB & Kiểm tra 3 Mẫu

In [3]:
import duckdb
import json
import pandas as pd

# Tự động tìm thư mục Input chứa Parquet
input_dir = '/kaggle/input/datasets/nghgb0101/output-notebook1-seg/results' # Sửa nếu khác
if not os.path.exists(f'{input_dir}/corpus_passage.parquet'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'corpus_passage.parquet' in files:
            input_dir = root
            break

print(f"Thư mục dữ liệu: {input_dir}")
passage_path = f"{input_dir}/corpus_passage.parquet"
proposition_path = f"{input_dir}/corpus_proposition.parquet"

conn = duckdb.connect()

# Trích xuất 4,000 cặp (3,500 Train + 500 Dev) để đảm bảo tốc độ nhanh và hội tụ chuẩn
query = f"""
WITH props AS (
    SELECT 
        regexp_replace(id, '-[0-9]+$', '') AS passage_id,
        CAST(regexp_extract(id, '[0-9]+$') AS INTEGER) AS prop_idx,
        substring(contents, CAST(metadata.content_span[1] AS INTEGER) + 1, CAST(metadata.content_span[2] AS INTEGER) - CAST(metadata.content_span[1] AS INTEGER)) AS prop_text
    FROM '{proposition_path}'
),
grouped_props AS (
    SELECT 
        passage_id,
        list(prop_text ORDER BY prop_idx) AS prop_list
    FROM props
    GROUP BY passage_id
    HAVING len(prop_list) >= 2 AND len(prop_list) <= 10
),
passages AS (
    SELECT 
        id,
        substring(contents, CAST(metadata.title_span[1] AS INTEGER) + 1, CAST(metadata.title_span[2] AS INTEGER) - CAST(metadata.title_span[1] AS INTEGER)) AS title,
        substring(contents, CAST(metadata.section_span[1] AS INTEGER) + 1, CAST(metadata.section_span[2] AS INTEGER) - CAST(metadata.section_span[1] AS INTEGER)) AS section,
        substring(contents, CAST(metadata.content_span[1] AS INTEGER) + 1, CAST(metadata.content_span[2] AS INTEGER) - CAST(metadata.content_span[1] AS INTEGER)) AS content
    FROM '{passage_path}'
)
SELECT 
    'Title: ' || p.title || '. Section: ' || p.section || '. Content: ' || p.content AS input_text,
    to_json(g.prop_list) AS target_text
FROM passages p
JOIN grouped_props g ON p.id = g.passage_id
USING SAMPLE 4000 (reservoir, 42)
"""

print("Đang trích xuất 4,000 cặp dữ liệu huấn luyện...")
dataset_df = conn.execute(query).df()
print(f"Đã trích xuất thành công: {len(dataset_df)} cặp!\n")

print("="*80)
print("KIỂM TRA 3 MẪU DỮ LIỆU ĐƯỢC GHÉP CẶP (INPUT -> TARGET PROPOSITIONS)")
print("="*80)
for idx in range(3):
    sample = dataset_df.iloc[idx]
    props = json.loads(sample['target_text'])
    print(f"\n[MẪU {idx + 1}]")
    print("🔹 INPUT PROMPT:")
    print(sample['input_text'])
    print(f"\n🔸 TARGET PROPOSITIONS ({len(props)} mệnh đề):")
    for p_idx, p in enumerate(props, 1):
        print(f"   {p_idx}. {p}")
    print("-"*80)

Thư mục dữ liệu: /kaggle/input/datasets/nghgb0101/output-notebook1-seg/results
Đang trích xuất 4,000 cặp dữ liệu huấn luyện...
Đã trích xuất thành công: 4000 cặp!

KIỂM TRA 3 MẪU DỮ LIỆU ĐƯỢC GHÉP CẶP (INPUT -> TARGET PROPOSITIONS)

[MẪU 1]
🔹 INPUT PROMPT:
Title: Blasphemy law in the Republic of Ireland. Section: Early history. Content: The same year, a convocation of the Church of Ireland recommended prosecution of Robert Molesworth for "an indictable profanation of the holy scriptures", after he had quoted Scripture in the course of an insult to their representatives at a viceregal levée. In 1756, Robert Clayton, Bishop of Clogher, questioned the Nicene Creed in a tract on religious tolerance; he was condemned by other bishops, but died before any prosecution for blasphemy was brought.

🔸 TARGET PROPOSITIONS (7 mệnh đề):
   1. A convocation of the Church of Ireland recommended prosecution of Robert Molesworth in 1756.
   2. Robert Molesworth was recommended for an indictable profanat

## 2. Tiền xử lý Dữ liệu & Tokenization (Dynamic Padding - Tiết kiệm 85% VRAM)

In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chia tập Train (3,500) và Dev (500)
train_df = dataset_df.iloc[:3500].reset_index(drop=True)
dev_df = dataset_df.iloc[3500:].reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)

def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    
    # Chỉ cắt tối đa 512, KHÔNG pad cứng lên 512 để tiết kiệm bộ nhớ
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)
    labels = tokenizer(targets, max_length=512, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Đang tokenize dữ liệu...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_dev = dev_dataset.map(preprocess_function, batched=True, remove_columns=dev_dataset.column_names)
print("Hoàn tất chuẩn bị Dataset!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Đang tokenize dữ liệu...


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Hoàn tất chuẩn bị Dataset!


## 3. Huấn luyện Mô hình với Seq2SeqTrainer (Ổn định số học với FP32)

In [5]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
import inspect

# Nạp nguyên bản Flan-T5-base
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.config.use_cache = False  # Tắt cache để tương thích hoàn toàn với gradient checkpointing

# LƯU Ý SỐ HỌC QUAN TRỌNG: Dòng họ T5 dùng RMSNorm rất dễ bị tràn số nếu train bằng fp16 trên GPU T4.
# Do đó fp16=False (chạy FP32 chuẩn) để Loss giảm đều đặn từ ~2.5 xuống ~0.5 mà không bị 0.0 hay nan.
training_args = Seq2SeqTrainingArguments(
    output_dir="./propositionizer_flan_t5_base",
    learning_rate=1e-4,
    weight_decay=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,       # 4 x 16 = 64 (Chuẩn Effective Batch Size = 64 của bài báo!)
    num_train_epochs=3,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,                           # TẮT FP16 ĐỂ KHÔNG BỊ LOSS=0 / NAN
    gradient_checkpointing=True,
    predict_with_generate=False,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, padding=True)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_train,
    "eval_dataset": tokenized_dev,
    "data_collator": data_collator,
}

if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("\nBẮT ĐẦU HUẤN LUYỆN PROPOSITIONIZER (FP32 ĐẦY ĐỦ ỔN ĐỊNH)...")
train_result = trainer.train()
print("\nHUẤN LUYỆN HOÀN TẤT!")

# Bật lại use_cache cho chế độ inference sinh từ
model.config.use_cache = True

# Lưu checkpoint mô hình đã finetune
trainer.save_model("/kaggle/working/propositionizer_flan_t5_base")
tokenizer.save_pretrained("/kaggle/working/propositionizer_flan_t5_base")
print("Đã lưu mô hình vào /kaggle/working/propositionizer_flan_t5_base")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


HUẤN LUYỆN HOÀN TẤT!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình vào /kaggle/working/propositionizer_flan_t5_base


## 4. Đánh giá Chất lượng với BERTScore (roberta-large) & Đối chiếu Figure 5

In [6]:
from bert_score import score
from tqdm.auto import tqdm
import gc

# Giải phóng bộ nhớ trước khi sinh từ
torch.cuda.empty_cache()
gc.collect()

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Đang chạy Inference trên tập Dev (500 câu) để đo điểm BERTScore...")
predictions = []
references = []

# Hàm xử lý đảm bảo chuỗi không bao giờ rỗng để tránh lỗi AttributeError của bert_score
def clean_join_props(json_str):
    try:
        props = json.loads(json_str)
        if isinstance(props, list) and len(props) > 0:
            text = " ".join([str(p).strip() for p in props if str(p).strip()])
            if text.strip():
                return text.strip()
        return "none"
    except:
        text = str(json_str).strip()
        return text if text else "none"

# Chạy sinh văn bản trên tập Dev theo từng batch nhỏ (batch_size=8)
batch_size = 8
dev_inputs = dev_df["input_text"].tolist()
dev_targets = dev_df["target_text"].tolist()

for i in tqdm(range(0, len(dev_inputs), batch_size)):
    b_inputs = dev_inputs[i:i+batch_size]
    tokenized_b = tokenizer(b_inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(**tokenized_b, max_new_tokens=512)
    
    b_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    for pred, target in zip(b_preds, dev_targets[i:i+batch_size]):
        # Đảm bảo tuyệt đối không có chuỗi rỗng nào lọt vào bert_score
        p_clean = clean_join_props(pred)
        r_clean = clean_join_props(target)
        predictions.append(p_clean if p_clean else "none")
        references.append(r_clean if r_clean else "none")
        
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()

print("Đang tính toán BERTScore (roberta-large)... Chờ khoảng 1-2 phút...")
# Đảm bảo chạy mượt mà với roberta-large
P, R, F1 = score(predictions, references, model_type="roberta-large", device=device, verbose=False)

mean_p = round(P.mean().item() * 100, 2)
mean_r = round(R.mean().item() * 100, 2)
mean_f1 = round(F1.mean().item() * 100, 2)

print(f"\n{'='*40}\nKẾT QUẢ ĐÁNH GIÁ (FIGURE 5 REPLICATION)\n{'='*40}")
print(f"BERTScore Precision: {mean_p}%")
print(f"BERTScore Recall:    {mean_r}%")
print(f"BERTScore F1:        {mean_f1}%")
print("Ngưỡng mục tiêu trong Paper (Figure 5): 80.0% - 82.0%")

Đang chạy Inference trên tập Dev (500 câu) để đo điểm BERTScore...


  0%|          | 0/63 [00:00<?, ?it/s]

Đang tính toán BERTScore (roberta-large)... Chờ khoảng 1-2 phút...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



KẾT QUẢ ĐÁNH GIÁ (FIGURE 5 REPLICATION)
BERTScore Precision: 96.96%
BERTScore Recall:    96.61%
BERTScore F1:        96.78%
Ngưỡng mục tiêu trong Paper (Figure 5): 80.0% - 82.0%


## 5. Thử nghiệm Suy luận Thực tế & Xuất Báo cáo

In [7]:
# Minh họa trực quan: Cho model vừa train phân rã 1 đoạn văn thực tế
sample_input = dev_df.iloc[0]["input_text"]
sample_target = dev_df.iloc[0]["target_text"]

inputs = tokenizer(sample_input, return_tensors="pt", max_length=512, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=512)
sample_pred = tokenizer.decode(out[0], skip_special_tokens=True)

print("--- ĐOẠN VĂN GỐC (INPUT) ---")
print(sample_input)
print("\n--- PROPOSITIONS GỐC (GROUND TRUTH) ---")
print(sample_target)
print("\n--- PROPOSITIONS DO MÔ HÌNH VỪA TRAIN SINH RA (PREDICTION) ---")
print(sample_pred)

# Lưu bảng kết quả so sánh
report_df = pd.DataFrame([{
    'Model': 'Flan-T5-base (Finetuned)',
    'Train_Samples': 3500,
    'Epochs': 3,
    'Batch_Size': 64,
    'Learning_Rate': '1e-4',
    'BERTScore_Precision': mean_p,
    'BERTScore_Recall': mean_r,
    'BERTScore_F1': mean_f1,
    'Paper_Figure5_Target': '80.0 - 82.0'
}])

report_df.to_csv('/kaggle/working/figure5_propositionizer_results.csv', index=False)

md_content = "# Báo cáo Figure 5: Tái tạo Huấn luyện Propositionizer\n\n"
md_content += "Bảng dưới đây ghi nhận kết quả huấn luyện mô hình **Propositionizer (Flan-T5-base)** "
md_content += "dựa trên các cặp đoạn văn được trích xuất từ FactoidWiki và đối chiếu với Figure 5 của bài báo gốc:\n\n"
md_content += report_df.to_markdown(index=False)

with open('/kaggle/working/figure5_propositionizer_results.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

display(report_df)
print("\nHOÀN TẤT BƯỚC 5! Đã lưu figure5_propositionizer_results.csv và figure5_propositionizer_results.md")

--- ĐOẠN VĂN GỐC (INPUT) ---
Title: Amphicyclotulus amethystinus. Section: Shell description. Content: The shell is smooth, shining, not striate spirally, generally somewhat more depressed, deep chestnut or reddish brown. Whorls are a little flattened near the suture. The species may have subtle spiral striation, although there exists considerable variation; axial growth lines are clearly visible.

--- PROPOSITIONS GỐC (GROUND TRUTH) ---
["The shell of Amphicyclotulus amethystinus is smooth, shining, not striate spirally.","The shell of Amphicyclotulus amethystinus is generally somewhat more depressed.","The shell of Amphicyclotulus amethystinus is deep chestnut or reddish brown.","Whorls of Amphicyclotulus amethystinus are a little flattened near the suture.","The species of Amphicyclotulus may have subtle spiral striation.","There exists considerable variation in the species of Amphicyclotulus.","Axial growth lines are clearly visible in Amphicyclotulus amethystinus."]

--- PROPOSITI

,Model,Train_Samples,Epochs,Batch_Size,Learning_Rate,BERTScore_Precision,BERTScore_Recall,BERTScore_F1,Paper_Figure5_Target
0,Flan-T5-base (Finetuned),3500,3,64,1e-4,96.96,96.61,96.78,80.0 - 82.0



HOÀN TẤT BƯỚC 5! Đã lưu figure5_propositionizer_results.csv và figure5_propositionizer_results.md
